# Sequence Filtering in PoolParty

This notebook demonstrates the filtering capabilities of PoolParty for designing high-quality oligonucleotide libraries.

## Overview

When designing DNA libraries for synthesis, it's important to filter out sequences that may cause:
- **Synthesis problems**: Homopolymers, extreme GC content
- **Cloning issues**: Unwanted restriction sites
- **Sequencing artifacts**: Low-complexity regions

PoolParty provides several filter methods:
- `filter_gc()` - Filter by GC content
- `filter_homopolymer()` - Remove sequences with long single-base runs
- `filter_complexity()` - Remove low-complexity (repetitive) sequences
- `filter_dust()` - Filter using the NCBI DUST algorithm
- `filter_restriction_sites()` - Remove sequences containing restriction enzyme sites

In [ ]:
import poolparty as pp

# Initialize poolparty
pp.init()

## Sequence Property Calculations

PoolParty provides standalone functions for calculating sequence properties. These can be used independently or as the basis for custom filters.

In [ ]:
# GC content calculation
print("GC Content Examples:")
print(f"  ATGC:     {pp.calc_gc('ATGC'):.2%}")
print(f"  GGGGCCCC: {pp.calc_gc('GGGGCCCC'):.2%}")
print(f"  AAAATTTT: {pp.calc_gc('AAAATTTT'):.2%}")

In [ ]:
# Complexity calculation (linguistic complexity)
print("Complexity Examples (0=repetitive, 1=complex):")
print(f"  ACGTMKWSRY:   {pp.calc_complexity('ACGTMKWSRY'):.3f}  (high complexity)")
print(f"  ACGTACGTACGT: {pp.calc_complexity('ACGTACGTACGT'):.3f}  (repetitive pattern)")
print(f"  AAAAAAAAAA:   {pp.calc_complexity('AAAAAAAAAA'):.3f}  (homopolymer)")

In [ ]:
# DUST score (lower = more complex)
print("DUST Score Examples (lower = more complex):")
print(f"  ACGTMKWSRYACGT: {pp.calc_dust('ACGTMKWSRYACGT'):.2f}")
print(f"  ATGATGATGATG:   {pp.calc_dust('ATGATGATGATG'):.2f}")
print(f"  AAAAAAAAAAAAA:  {pp.calc_dust('AAAAAAAAAAAAA'):.2f}")

In [ ]:
# Homopolymer detection
print("Homopolymer Detection:")
print(f"  ACGTAAAAAACGT has run > 4: {pp.has_homopolymer('ACGTAAAAAACGT', 4)}")
print(f"  ACGTAAAACGT has run > 4:   {pp.has_homopolymer('ACGTAAAACGT', 4)}")

## GC Content Filtering

Filter sequences to maintain GC content within a target range. This is important for:
- Uniform melting temperatures
- Balanced synthesis efficiency
- Avoiding secondary structures

In [ ]:
# Create a pool with varying GC content
with pp.Party():
    seqs = [
        "GGGGGGGGGGGG",  # 100% GC
        "AAAAAAAAAAAA",  # 0% GC
        "ATGCATGCATGC",  # 50% GC
        "GCGCATATATAT",  # 33% GC
        "GCGCGCATATAT",  # 50% GC
    ]
    
    root = pp.from_seqs(seqs, mode="sequential")
    
    # Filter to keep only 40-60% GC
    filtered = root.filter_gc(min_gc=0.4, max_gc=0.6)
    
    df = filtered.generate_library(num_seqs=5, discard_null_seqs=True)
    
    print(f"Original sequences: {len(seqs)}")
    print(f"After GC filter (40-60%): {len(df)}")
    print("\nFiltered sequences:")
    for seq in df['seq']:
        print(f"  {seq} - GC: {pp.calc_gc(seq):.0%}")

## Homopolymer Filtering

Long runs of the same base (homopolymers) can cause:
- Synthesis errors (especially poly-G and poly-C)
- Sequencing errors (especially in nanopore)
- PCR slippage

In [ ]:
with pp.Party():
    seqs = [
        "ACGTACGTACGT",      # No homopolymers
        "ACGTAAAAAACGT",     # 6 A's
        "ACGTGGGGGGACGT",    # 6 G's
        "ACGTAAAACGT",       # 4 A's (allowed)
        "AACGTAACGTAACGT",   # Only 2-mers (allowed)
    ]
    
    root = pp.from_seqs(seqs, mode="sequential")
    
    # Allow runs up to 4 bases, filter out longer ones
    filtered = root.filter_homopolymer(max_length=4)
    
    df = filtered.generate_library(num_seqs=5, discard_null_seqs=True)
    
    print(f"Original sequences: {len(seqs)}")
    print(f"After homopolymer filter (max 4): {len(df)}")
    print("\nFiltered sequences:")
    for seq in df['seq']:
        print(f"  {seq}")

## Complexity Filtering

Low-complexity sequences (repetitive patterns) can cause:
- Alignment ambiguities
- Difficulty in primer design
- Recombination during cloning

In [ ]:
with pp.Party():
    seqs = [
        "ACGTMKWSRYACGT",    # High complexity
        "ACGTACGTACGTACGT",  # Repetitive pattern
        "ATATATATATATAT",    # Dinucleotide repeat
        "AAAAAAAAAAAAA",     # Homopolymer (very low)
        "TGCAMKWSRYTGCA",    # High complexity
    ]
    
    root = pp.from_seqs(seqs, mode="sequential")
    
    # Keep only sequences with complexity >= 0.5
    filtered = root.filter_complexity(min_complexity=0.5)
    
    df = filtered.generate_library(num_seqs=5, discard_null_seqs=True)
    
    print(f"Original sequences: {len(seqs)}")
    print(f"After complexity filter (>= 0.5): {len(df)}")
    print("\nFiltered sequences with complexity scores:")
    for seq in df['seq']:
        print(f"  {seq} - complexity: {pp.calc_complexity(seq):.3f}")

## DUST Score Filtering

The DUST algorithm is used by NCBI BLAST for masking low-complexity regions. Lower scores indicate higher complexity.

In [ ]:
with pp.Party():
    seqs = [
        "ACGTMKWSRYACGT",    # Complex (low DUST)
        "ATGATGATGATG",      # Triplet repeat (medium DUST)
        "AAAAAAAAAAAAA",     # Homopolymer (high DUST)
        "TGCATGCATGCA",      # Some pattern (low DUST)
    ]
    
    root = pp.from_seqs(seqs, mode="sequential")
    
    # Keep sequences with DUST score <= 2.0 (more complex)
    filtered = root.filter_dust(max_score=2.0)
    
    df = filtered.generate_library(num_seqs=4, discard_null_seqs=True)
    
    print(f"Original sequences: {len(seqs)}")
    print(f"After DUST filter (<= 2.0): {len(df)}")
    print("\nFiltered sequences with DUST scores:")
    for seq in df['seq']:
        print(f"  {seq} - DUST: {pp.calc_dust(seq):.2f}")

## Restriction Site Filtering

When designing sequences for cloning, you need to avoid restriction sites that would interfere with your cloning strategy.

PoolParty includes a database of ~100 common restriction enzymes and several presets:
- `golden_gate` - BsaI, BsmBI, BbsI, etc.
- `common` - EcoRI, BamHI, HindIII, etc.
- `mcs` - Standard multiple cloning site enzymes
- `frequent_cutters` - 4-base cutters like DpnI
- `rare_cutters` - 8-base cutters like NotI

In [ ]:
# View available presets
print("Available enzyme presets:")
for preset_name, enzymes in pp.ENZYME_PRESETS.items():
    print(f"  {preset_name}: {', '.join(enzymes[:5])}{'...' if len(enzymes) > 5 else ''}")

In [ ]:
# Look up individual enzyme sites
print("Enzyme recognition sites:")
for enzyme in ["EcoRI", "BamHI", "BsaI", "NotI"]:
    print(f"  {enzyme}: {pp.get_enzyme_site(enzyme)}")

In [ ]:
with pp.Party():
    seqs = [
        "ACGTACGTACGT",       # No restriction sites
        "ACGTGAATTCACGT",     # Contains EcoRI (GAATTC)
        "ACGTGGATCCACGT",     # Contains BamHI (GGATCC)
        "ACGTTGCAACGT",       # No common sites
        "ACGTAAGCTTACGT",     # Contains HindIII (AAGCTT)
    ]
    
    root = pp.from_seqs(seqs, mode="sequential")
    
    # Filter out sequences with EcoRI, BamHI, or HindIII sites
    filtered = root.filter_restriction_sites(enzymes=["EcoRI", "BamHI", "HindIII"])
    
    df = filtered.generate_library(num_seqs=5, discard_null_seqs=True)
    
    print(f"Original sequences: {len(seqs)}")
    print(f"After restriction site filter: {len(df)}")
    print("\nFiltered sequences (no EcoRI/BamHI/HindIII):")
    for seq in df['seq']:
        print(f"  {seq}")

In [ ]:
# Using presets for Golden Gate cloning
with pp.Party():
    seqs = [
        "ACGTACGTACGT",       # No sites
        "ACGTGGTCTCACGT",     # Contains BsaI (GGTCTC)
        "ACGTCGTCTCACGT",     # Contains BsmBI (CGTCTC)
        "ACGTTGCAACGT",       # No Golden Gate sites
    ]
    
    root = pp.from_seqs(seqs, mode="sequential")
    
    # Use the golden_gate preset to filter all Golden Gate enzyme sites
    filtered = root.filter_restriction_sites(enzymes=["golden_gate"])
    
    df = filtered.generate_library(num_seqs=4, discard_null_seqs=True)
    
    print(f"Original sequences: {len(seqs)}")
    print(f"After Golden Gate filter: {len(df)}")
    print("\nSafe for Golden Gate cloning:")
    for seq in df['seq']:
        print(f"  {seq}")

## Combining Multiple Filters

Filters can be chained together to apply multiple quality criteria. Each filter passes sequences that meet its criteria to the next filter.

In [ ]:
with pp.Party():
    # Generate a diverse set of sequences
    seqs = [
        "ATGCATGCATGC",       # Good: 50% GC, no homopolymer, no sites
        "GGGGGGGGGGGG",       # Bad: 100% GC
        "ACGTAAAAAACGT",      # Bad: homopolymer
        "ACGTGAATTCACGT",     # Bad: EcoRI site
        "AAAATTTTAAAA",       # Bad: 0% GC + homopolymer
        "TGCAMKWSRYTGCA",     # Good: diverse, no issues
        "GCATATGCATAT",       # Good: balanced
    ]
    
    root = pp.from_seqs(seqs, mode="sequential")
    
    # Apply multiple filters in sequence
    filtered = (
        root
        .filter_gc(min_gc=0.3, max_gc=0.7)           # 30-70% GC
        .filter_homopolymer(max_length=4)            # No runs > 4
        .filter_restriction_sites(enzymes=["EcoRI"]) # No EcoRI
    )
    
    df = filtered.generate_library(num_seqs=7, discard_null_seqs=True)
    
    print(f"Original sequences: {len(seqs)}")
    print(f"After all filters: {len(df)}")
    print("\nHigh-quality sequences:")
    for seq in df['seq']:
        gc = pp.calc_gc(seq)
        print(f"  {seq} - GC: {gc:.0%}")

## Practical Example: Library Design for MPRA

Let's design a library of regulatory element variants with quality filters applied.

In [ ]:
with pp.Party():
    # Start with a template regulatory element
    template = pp.from_seq("ACGTGCTAGCATGCTAGCTACGATCG")
    
    # Apply mutagenesis to generate variants
    variants = template.mutagenize(
        num_mutations=2,
        mode="random"
    )
    
    # Apply quality filters
    filtered = (
        variants
        .filter_gc(min_gc=0.4, max_gc=0.6)           # Maintain GC balance
        .filter_homopolymer(max_length=5)            # Avoid synthesis issues
        .filter_restriction_sites(enzymes=["common"]) # Avoid cloning sites
    )
    
    # Generate filtered library
    df = filtered.generate_library(
        num_seqs=20,
        discard_null_seqs=True,
        seed=42
    )
    
    print(f"Generated {len(df)} high-quality variants")
    print("\nSample variants:")
    for i, row in df.head(5).iterrows():
        seq = row['seq']
        gc = pp.calc_gc(seq)
        print(f"  {seq} (GC: {gc:.0%})")

## Summary

PoolParty's filtering capabilities help ensure your designed libraries are:

1. **Synthesizable**: Appropriate GC content, no problematic homopolymers
2. **Clonable**: No unwanted restriction sites
3. **High-quality**: Adequate sequence complexity

Key functions:
- `calc_gc()`, `calc_complexity()`, `calc_dust()` - Standalone property calculations
- `filter_gc()` - GC content range filtering
- `filter_homopolymer()` - Homopolymer run length filtering
- `filter_complexity()` - Linguistic complexity filtering
- `filter_dust()` - DUST score filtering
- `filter_restriction_sites()` - Restriction enzyme site filtering